In [1]:
import os
import pandas as pd

In [10]:
# 根据af3的输出结果和rmsd，统计所有rmsd大于10的complex占用空间大小
af3_output_dir = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/3_AF3/predict/outputs'
metrics_file = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/3_AF3/predict/metrices/metrics_summary.csv'
total_size = 0
json_total_size = 0
json_summary_total_size = 0
sample_num = 0
cutoff = 0.0

df = pd.read_csv(metrics_file)
for _, row in df.iterrows():
    if row['scRMSD'] > cutoff:
        complex = row['complex']
        seed = row['seed']
        id = row['id']
        complex_dir = af3_output_dir + "/" + complex + "/" + f"seed-{seed}_sample-{id}"
        file = complex_dir + f"/{complex}_seed-{seed}_sample-{id}_model.cif"
        json_file = complex_dir + f"/{complex}_seed-{seed}_sample-{id}_confidences.json"
        json_summary_file = complex_dir + f"/{complex}_seed-{seed}_sample-{id}_summary_confidences.json"
        if os.path.exists(file):
            size = os.path.getsize(file)
            json_size = os.path.getsize(json_file)
            json_summary_size = os.path.getsize(json_summary_file)
            total_size += size
            json_total_size += json_size
            json_summary_total_size += json_summary_size
            sample_num += 1

print(f'Total size of complexes with scRMSD > {cutoff}: {total_size / (1024 * 1024 * 1024):.2f} GB')
print(f'Total size of confidence JSON files with scRMSD > {cutoff}: {json_total_size / (1024 * 1024 * 1024):.2f} GB')
print(f'Total size of summary confidence JSON files with scRMSD > {cutoff}: {json_summary_total_size / (1024 * 1024 * 1024):.2f} GB')
print(f'Number of samples with scRMSD > {cutoff}: {sample_num}')

Total size of complexes with scRMSD > 0.0: 28.98 GB
Total size of confidence JSON files with scRMSD > 0.0: 102.79 GB
Total size of summary confidence JSON files with scRMSD > 0.0: 0.06 GB
Number of samples with scRMSD > 0.0: 202150


In [ ]:
import gzip, os

target = "/home/junjiechen/archive/250401-Dpepalign/Benchmark/Rosetta/AF3/predict/outputs-done"

n = 0
for dirpath, _, files in os.walk(target):
    for f in files:
        if not f.endswith("_confidences.json") or f.endswith("_summary_confidences.json"):
            continue
        src = os.path.join(dirpath, f)
        dst = src + ".gz"
        if os.path.exists(dst):
            continue
        with open(src, "rb") as fi:
            data = fi.read()
        with gzip.open(dst, "wb", compresslevel=6) as fo:
            fo.write(data)
        os.remove(src)
        n += 1
        if n % 10000 == 0:
            print(f"Compressed {n} files...", flush=True)

print(f"Done. Compressed {n} files.")

In [9]:
# 检查/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/3_AF3/predict/metrices/metrics_summary.csv和/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/3_AF3/predict/outputs/metrics_summary.csv除了iPAE列之外的所有列是否一致
metrics_summary_file_1 = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/3_AF3/predict/metrices/metrics_summary.csv'
metrics_summary_file_2 = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface-classifier/3_AF3/predict/outputs/metrics_summary.csv'
df1 = pd.read_csv(metrics_summary_file_1)
df2 = pd.read_csv(metrics_summary_file_2)
columns_to_check = [col for col in df1.columns if col != 'iPAE']
for col in columns_to_check:
    if not df1[col].equals(df2[col]):
        print(f'Column {col} is different between the two files.')
    else:
        print(f'Column {col} is the same between the two files.')

Column complex is the same between the two files.
Column seed is the same between the two files.
Column id is the same between the two files.
Column pep_plDDT is the same between the two files.
Column plDDT is the same between the two files.
Column ipTM is the same between the two files.
Column pTM is the same between the two files.
Column pep_pTM is the same between the two files.
Column pAE_min is the same between the two files.
Column ipAE is different between the two files.
Column ranking_score is the same between the two files.
Column scRMSD is the same between the two files.
